In [22]:
import pandas as pd
import folium
import pgeocode

# Load your data
departments = pd.read_csv("departments.csv")
patients = pd.read_csv("patients.csv")

# Filter out placeholder rows (-2, -3)
dept = departments[departments["DepartmentKey"] > 0].copy()

# Geocode by ZIP code instantly (no API, no rate limit)
nomi = pgeocode.Nominatim('us')
results = nomi.query_postal_code(dept["PostalCode"].astype(str).tolist())
dept["lat"] = results["latitude"].values
dept["lon"] = results["longitude"].values

# Drop any ZIPs that couldn't be found
dept = dept.dropna(subset=["lat", "lon"])

# Merge patient counts into departments
if "DepartmentKey" in patients.columns:
    pat_counts = patients.groupby("DepartmentKey").size().reset_index(name="patient_count")
    dept = dept.merge(pat_counts, on="DepartmentKey", how="left")
    dept["patient_count"] = dept["patient_count"].fillna(0)
else:
    dept["patient_count"] = 1  # uniform size if no patient link

# Build the map centered on your data
map_center = [dept["lat"].mean(), dept["lon"].mean()]
m = folium.Map(location=map_center, zoom_start=8)

# Plot each department as a bubble
for _, row in dept.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=max(5, row["patient_count"] ** 0.5),  # scale bubble by sqrt of count
        color="#1D9E75",
        fill=True,
        fill_opacity=0.7,
        tooltip=(
            f"Department: {row['DepartmentName']}\n"
            f"City: {row['City']}\n"
            f"County: {row['County']}\n"
            f"Specialty: {row['DepartmentSpecialty']}\n"
            f"Type: {row['DepartmentType']}\n"
            f"Patients: {int(row['patient_count'])}"
        )
    ).add_to(m)

# Save and confirm
m.save("departments_map.html")
print(f"Map saved → departments_map.html")
print(f"Departments plotted: {len(dept)}")
print(f"Total patients mapped: {int(dept['patient_count'].sum())}")

Map saved → departments_map.html
Departments plotted: 7018
Total patients mapped: 7018


In [ ]:
#print files


department.tail()

,DepartmentKey,Address,City,County,DepartmentName,DepartmentSpecialty,DepartmentType,PostalCode,CensusTract
11592,12908,909 SW Mulvane St,TOPEKA,SHAWNEE,STORMONT VAIL ALLERGY CLINIC,Allergy,*Unknown,66606,2.017700e+10
11593,12909,1414 SW 8th Ave,Topeka,*Unspecified,CC HEMATOLOGY ONCOLOGY,Hematology and Oncology,*Unknown,66606,2.017700e+10
11594,12910,NaN,NaN,*Unspecified,*Unspecified,*Unspecified,*Unspecified,NaN,NaN
11595,12911,1500 SW 10th Ave,Topeka,SHAWNEE,SV PHARMACOTHERAPY,Pharmacy,HOD,66606,2.017700e+10
11596,12912,330 SW Oakley,TOPEKA,SHAWNEE,ASTRA MENTAL HEALTH & RECOVERY,*Not Applicable,*Not Applicable,66606,2.017700e+10


In [16]:
import requests

geojson = requests.get(
    "https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json"
).json()

In [17]:
import plotly.express as px
import pandas as pd

geojson = px.data.election_geojson()

df = pd.DataFrame({
    "fips": ["36001", "36005"],
    "avg_delay": [14, 9]
})

fig = px.choropleth(
    df,
    geojson=geojson,
    locations="fips",
    color="avg_delay",
    featureidkey="properties.GEOID",
    color_continuous_scale="Reds"
)

fig.update_geos(fitbounds="locations", visible=False)
fig.show()